# Modeling Pipeline

In [ ]:
# Load and clean data (same pipeline as EDA)
df = pd.read_csv(Path.cwd().parent / ".temp" / "vehicles.csv")

drop_cols = [
    "id", "url", "region_url", "image_url", "description", "county",
    "VIN", "posting_date", "region", "model", "size", "state", "lat", "long",
]
df = df.drop(columns=drop_cols)

df = df[(df["price"] >= 500) & (df["price"] <= 80_000)]
df = df[(df["year"] >= 1990) & (df["year"] <= 2021)]
df = df[(df["odometer"] > 0) & (df["odometer"] <= 400_000)]
df = df.dropna(subset=["year", "manufacturer", "fuel", "odometer", "title_status", "transmission"])

df["cylinders_num"] = df["cylinders"].str.extract(r"(\d+)").astype(float)
df["vehicle_age"] = 2021 - df["year"]

for col in ["condition", "drive", "type", "paint_color"]:
    df[col] = df[col].fillna("unknown")
df["cylinders_num"] = df["cylinders_num"].fillna(df["cylinders_num"].median())

numeric_features = ["vehicle_age", "odometer", "cylinders_num"]
categorical_features = ["manufacturer", "fuel", "title_status", "transmission", "drive", "type", "paint_color", "condition"]

df_model = df[numeric_features + categorical_features + ["price"]].dropna()
X = df_model[numeric_features + categorical_features]
y = df_model["price"]
print(f"Final dataset: {df_model.shape}")

## Train / Validation / Test Split

In [ ]:
df = pd.read_csv('../.temp/vehicles.csv')

df = df[(df['price'] >= 500) & (df['price'] <= 80000)]
df = df[(df['year'] >= 1990) & (df['year'] <= 2021)]
df = df[(df['odometer'] > 0) & (df['odometer'] <= 400000)]

drop_cols = ['id', 'url', 'region_url', 'image_url', 'description',
             'county', 'VIN', 'posting_date', 'region', 'model', 'size', 'state', 'lat', 'long']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df = df.dropna(subset=['year', 'odometer', 'price'])

cyl_map = {'3 cylinders': 3, '4 cylinders': 4, '5 cylinders': 5,
           '6 cylinders': 6, '8 cylinders': 8, '10 cylinders': 10, '12 cylinders': 12}
df['cylinders_num'] = df['cylinders'].map(cyl_map)
df['cylinders_num'] = df['cylinders_num'].fillna(df['cylinders_num'].median())
df = df.drop(columns=['cylinders'])

df['vehicle_age'] = 2021 - df['year']
df = df.drop(columns=['year'])

categorical_features = ['manufacturer', 'fuel', 'title_status', 'transmission', 'drive', 'type', 'paint_color', 'condition']
for col in categorical_features:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

print(f'Dataset shape: {df.shape}')

In [ ]:
numeric_features = ['vehicle_age', 'odometer', 'cylinders_num']

X = df.drop(columns=['price'])
y = df['price']

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.15, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.176, random_state=RANDOM_STATE)

print(f'Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}')

iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE)
outlier_preds = iso.fit_predict(X_train[numeric_features])
mask = outlier_preds == 1
print(f'Outliers removed from training set: {(~mask).sum()}')
X_train_clean = X_train[mask]
y_train_clean = y_train[mask]

## Preprocessing Pipeline

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=100, sparse_output=False), categorical_features)
])

## Dummy Baseline

In [ ]:
dummy_pipe = Pipeline([('preprocessor', preprocessor), ('model', DummyRegressor(strategy='median'))])
dummy_pipe.fit(X_train_clean, y_train_clean)

for name, Xs, ys in [('Val', X_val, y_val), ('Test', X_test, y_test)]:
    preds = dummy_pipe.predict(Xs)
    print(f'{name} - MAE: {mean_absolute_error(ys, preds):.2f}, RMSE: {root_mean_squared_error(ys, preds):.2f}, R2: {r2_score(ys, preds):.4f}')

dummy_test_preds = dummy_pipe.predict(X_test)
dummy_mae = mean_absolute_error(y_test, dummy_test_preds)
dummy_rmse = root_mean_squared_error(y_test, dummy_test_preds)
dummy_r2 = r2_score(y_test, dummy_test_preds)

## Ridge Regression Baseline

In [ ]:
ridge_pipe = Pipeline([('preprocessor', preprocessor), ('model', Ridge(alpha=1.0))])
ridge_pipe.fit(X_train_clean, y_train_clean)

for name, Xs, ys in [('Val', X_val, y_val), ('Test', X_test, y_test)]:
    preds = ridge_pipe.predict(Xs)
    print(f'{name} - MAE: {mean_absolute_error(ys, preds):.2f}, RMSE: {root_mean_squared_error(ys, preds):.2f}, R2: {r2_score(ys, preds):.4f}')

ridge_test_preds = ridge_pipe.predict(X_test)
ridge_mae = mean_absolute_error(y_test, ridge_test_preds)
ridge_rmse = root_mean_squared_error(y_test, ridge_test_preds)
ridge_r2 = r2_score(y_test, ridge_test_preds)

## XGBoost with Optuna Hyperparameter Optimization

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train_clean)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': RANDOM_STATE,
        'n_jobs': -1
    }
    model = XGBRegressor(**params)
    model.fit(X_train_processed, y_train_clean)
    preds = model.predict(X_val_processed)
    return mean_absolute_error(y_val, preds)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=60)

print(f'Best MAE: {study.best_value:.2f}')
print(f'Best params: {study.best_params}')

In [ ]:
best_xgb = XGBRegressor(**study.best_params, random_state=RANDOM_STATE, n_jobs=-1)
best_xgb.fit(X_train_processed, y_train_clean)

for name, Xp, ys in [('Val', X_val_processed, y_val), ('Test', X_test_processed, y_test)]:
    preds = best_xgb.predict(Xp)
    print(f'{name} - MAE: {mean_absolute_error(ys, preds):.2f}, RMSE: {root_mean_squared_error(ys, preds):.2f}, R2: {r2_score(ys, preds):.4f}')

xgb_test_preds = best_xgb.predict(X_test_processed)
xgb_mae = mean_absolute_error(y_test, xgb_test_preds)
xgb_rmse = root_mean_squared_error(y_test, xgb_test_preds)
xgb_r2 = r2_score(y_test, xgb_test_preds)

## Results Summary

In [ ]:
pd.DataFrame(results).set_index("Model")

## Feature Importance

In [ ]:
feature_names = preprocessor.get_feature_names_out()
importances = best_xgb.feature_importances_

top_k = 15
top_idx = np.argsort(importances)[-top_k:]
top_names = [feature_names[i].replace("cat__", "").replace("num__", "") for i in top_idx]
top_values = importances[top_idx]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(top_k), top_values, color="steelblue")
ax.set_yticks(range(top_k))
ax.set_yticklabels(top_names)
ax.set_xlabel("Feature Importance (split count)")
ax.set_title("Top 15 XGBoost Feature Importances")
fig.tight_layout()
plt.show()